# VIA 人工标注复核

## tl;dr

三个事件共 243 张图片、43 个独立人工粗框。West 0～+2 分钟肉眼不可分辨；后段 77 帧省略补框的参考是高分辨率组。空框不能作为无烟标签。当前标签可支持早期裁剪诊断，其他烟雾和困难背景尚未完整编码。

## Context & Methods

这是标注与几何复核伴随 notebook，不是新一轮模型实验。VIA 原文件、采集清单、原图和第四次保存输出是输入；精确计算位于 `scripts/audit_annotations.py`。

### Key Assumptions

- 阶段语义采用标注者 2026-10-21 的说明，后段确认参考高分辨率组。
- 人工框是粗略总范围，不是逐像素分割；覆盖比例只衡量矩形几何。
- 人工框只有 43 张，四组复用，不作 172 张独立样本。
- 不用模型框补造几何真值，不重新推理。

### 1. 定位输入与计算入口

仅使用 Python 标准库。可从仓库根目录或本 notebook 目录启动；下面的单元按顺序执行，不写源数据。

In [1]:
from pathlib import Path
import ast, json, runpy, random, sys
sys.dont_write_bytecode = True
candidates = [Path.cwd(), *Path.cwd().parents, Path('D:/AIC/AIC-seek')]
root = next(p for p in candidates if (p / 'experiments/exp004_figlib_multi_event/events.json').is_file())
script = root / 'experiments/exp004_figlib_multi_event/scripts/audit_annotations.py'
analysis = runpy.run_path(str(script))
labels, audit = analysis['build_audit'](root, coverage_reference='high_resolution')
print('输入复核完成；未加载权重或运行模型。')

输入复核完成；未加载权重或运行模型。


## Data

### 2. 检查覆盖范围与语义

每张图片按事件目录及文件名唯一匹配，同时核验原图哈希、VIA 坐标与保存预测的时间轴。表格为完整的三个事件汇总。

In [2]:
print('事件       总帧数   人工框帧数   首次可辨识/分钟   最后人工框/分钟')
for event in audit['events']:
    print(f"{event['event']:<10} {event['frames']:>3} {event['manual_box_frames']:>10} {event['first_human_discernible_offset_seconds']//60:>16} {event['last_manual_box_offset_seconds']//60:>16}")
print('\n定位状态:', json.dumps(audit['totals']['localization_counts'], ensure_ascii=False))
print('保存预测步数:', audit['totals']['audited_prediction_steps'])
print('省略补框参考:', audit['coverage_reference_setting'])

事件       总帧数   人工框帧数   首次可辨识/分钟   最后人工框/分钟
alison      81         17                0               16
west        81         20                3               22
cranston    81          6                0                5

定位状态: {"not_applicable": 120, "manual_coarse_box": 43, "omitted_model_covered": 77, "not_localizable": 3}
保存预测步数: 972
省略补框参考: high_resolution


## Results

### 3. 检查早期真实裁剪

按原模型的扩张、补方和边界裁切规则重建像素范围。下面仅选出与人工框有交集的保留轨迹作为几何核查对象；交集本身不赋予正式真阳性身份。

In [3]:
selected = {}
for event, minute in [('alison', 3), ('cranston', 2)]:
    item = next(row for row in audit['geometry_evidence'] if row['event'] == event and row['setting'] == 'full_baseline' and row['offset_seconds'] == minute*60)
    tubes = [tube for tube in item['kept_with_current_real_observation'] if tube['current_box_overlap']['iou'] > 0]
    assert len(tubes) == 1
    tube = tubes[0]
    selected[event] = tube
    historical = tube['annotated_historical_frames']
    print(f"{event} +{minute}分 | crop={tube['crop_xyxy']} | 人工框覆盖={tube['crop_reference_box_coverage']:.1%} | L={tube['logit']:.6f} | p={tube['probability']:.6f}")
    print('  不同真实观测数:', tube['independent_observed_frames'], '; 历史人工框覆盖:', [round(v['crop_reference_box_coverage'], 6) for v in historical])

alison +3分 | crop=[2712, 982, 2753, 1024] | 人工框覆盖=100.0% | L=0.344259 | p=0.027026
  不同真实观测数: 2 ; 历史人工框覆盖: [1.0, 1.0]
cranston +2分 | crop=[705, 683, 797, 775] | 人工框覆盖=100.0% | L=-5.488263 | p=0.001862
  不同真实观测数: 2 ; 历史人工框覆盖: [1.0, 1.0]


这两个窗口已经覆盖人工粗框，却尚未形成足够的最终判定证据。“完全裁掉烟羽”的解释受到限制，尺度、上下文与时序证据仍需控制实验。它不证明结构改进一定有效。

### 4. 粗框与局部检测的区别

West +5 分钟的局部候选落在烟羽粗范围内，但 IoU 低于 0.5。仅用一个 IoU 门槛会改变原因诊断对象。

In [4]:
item = next(row for row in audit['geometry_evidence'] if row['event']=='west' and row['setting']=='full_baseline' and row['offset_seconds']==300)
candidate = max(item['candidates'], key=lambda row: row['iou'])
print(f"West +5分：IoU={candidate['iou']:.6f}；检测框落在人工框内的比例={candidate['prediction_inside_reference_fraction']:.1%}；人工粗框覆盖={candidate['reference_box_coverage']:.1%}")

West +5分：IoU=0.239582；检测框落在人工框内的比例=100.0%；人工粗框覆盖=24.0%


### 5. 防止标签误读与几何实现漂移

检查内部断档、完全缺标及前后段语义；用本机原模型的纯几何函数对照当前计算。此检查不导入 PyTorch，也不执行模型推理。

In [5]:
interpret = analysis['interpret_empty']
assert interpret(0,180,1320)[:2] == ('not_discernible','not_localizable')
assert interpret(1380,180,1320)[:2] == ('visible','omitted_model_covered')
assert interpret(600,180,1320)[:2] == ('unresolved','unresolved')
assert interpret(0,None,None)[:2] == ('unresolved','unresolved')
assert interpret(-60,180,1320)[0] == 'before_official_onset'
assert len(labels)==243 and sum(row['geometry_evaluable'] for row in labels)==43
assert len(audit['geometry_evidence'])==43*4
assert all(row['coverage_reference_setting']=='high_resolution' for row in labels if row['localization_status']=='omitted_model_covered')
assert all(not row['geometry_evaluable'] and not row['manual_boxes_xyxy'] for row in labels if row['localization_status']=='omitted_model_covered')
for tube in selected.values():
    assert tube['independent_observed_frames']==2
    assert len(tube['annotated_historical_frames'])==2
    assert all(row['crop_reference_box_coverage']==1.0 for row in tube['annotated_historical_frames'])
source = root.parent/'pyronear_lab/temporal-model/core/src/temporal_model/core/crop.py'
tree = ast.parse(source.read_text(encoding='utf-8'))
functions = [node for node in tree.body if isinstance(node, ast.FunctionDef) and node.name in ('expand_bbox','norm_bbox_to_pixel_square')]
assert len(functions)==2
geometry = {}
exec(compile(ast.Module(body=functions, type_ignores=[]),str(source),'exec'),geometry)
rng = random.Random(20261021)
for _ in range(1000):
    width,height=rng.choice([(3072,2048),(2048,1536)])
    window=[rng.random(),rng.random(),rng.uniform(.002,.4),rng.uniform(.002,.4)]
    factor=rng.choice([1.0,1.5,2.0])
    original=geometry['norm_bbox_to_pixel_square'](*geometry['expand_bbox'](*window,factor),width,height)
    assert tuple(analysis['crop_box'](window,width,height,factor))==original
print('语义边界检查通过；1,000 组裁剪几何与原实现一致。')

语义边界检查通过；1,000 组裁剪几何与原实现一致。


## Takeaways

现有 43 个早期框可以直接支持第五次裁剪原因定位。West 的不可辨识时段、后段省略补框和出现前目标未到官方起点是三种不同状态。补充工作集中在 West 困难背景身份/时间，以及其他设置后段关键轨迹身份；无需重新绘制全部图片。正式模型成绩须区分目标候选、保留轨迹、跨阈、最高分归属与实际告警。

共享标准见 `../../ANNOTATION_PROTOCOL.md`，共享标签与来源哈希见同目录的 `frame_labels.jsonl`、`audit.json`。当前未计算正式 mAP、总体误报率或新事件泛化成绩。